In [28]:
import copy
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

In [29]:
# 0) Setup
# ------------------------------------------------------------

torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

student_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

student_model.train()
for p in student_model.parameters():
    p.requires_grad_(True)

# Paper-style fixed initial teacher.
teacher_model = copy.deepcopy(student_model).to(device)
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad_(False)

optimizer = torch.optim.Adam(student_model.parameters(), lr=1e-5)

print("Loaded:", model_name, "on", device)

Loaded: sshleifer/tiny-gpt2 on cpu


In [30]:
def make_student_prompt(problem):
    return (
        "Problem: " + problem + "\n"
        "Answer:"
    )


def make_teacher_prompt(problem, solution):
    return (
        "Problem: " + problem + "\n"
        "Here is a reference solution:\n"
        + solution + "\n"
        "After understanding the reference solution, please try to solve this problem "
        "using your own approach below:\n"
        "Answer:"
    )

In [31]:
def first_eos_lengths(response_ids, eos_token_id):
    """
    response_ids: [B, T]

    Returns:
        response_lens: [B]

    Length includes the first EOS token.
    If no EOS appears, length = T.
    """
    B, T = response_ids.shape

    eos_mask = response_ids.eq(eos_token_id)  # [B, T]

    has_eos = eos_mask.any(dim=1)  # [B]

    first_eos_idx = eos_mask.float().argmax(dim=1).long()  # [B]

    response_lens = torch.where(
        has_eos,
        first_eos_idx + 1,
        torch.full((B,), T, device=response_ids.device, dtype=torch.long),
    )

    return response_lens

In [32]:
@torch.no_grad()
def generate_student_rollouts(
    model,
    tokenizer,
    student_prompts,
    max_new_tokens=32,
    temperature=1.0,
    top_k=50,
):
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    batch = tokenizer(
        student_prompts,
        return_tensors="pt",
        padding=True,
        add_special_tokens=False,
    )

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    prompt_width = input_ids.shape[1]

    was_training = model.training
    model.eval()

    output_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        do_sample=True,
        temperature=temperature,
        top_k=top_k,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    if was_training:
        model.train()

    tokenizer.padding_side = old_padding_side

    raw_response_ids = output_ids[:, prompt_width:]
    response_lens = first_eos_lengths(raw_response_ids, tokenizer.eos_token_id)

    response_ids_list = [
        raw_response_ids[i, : response_lens[i]].detach().cpu()
        for i in range(raw_response_ids.shape[0])
    ]

    return response_ids_list


In [33]:
def tokenize_prompts_no_padding(tokenizer, prompts):
    prompt_ids_list = []

    for prompt in prompts:
        ids = tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False,
        )["input_ids"][0]

        prompt_ids_list.append(ids)

    return prompt_ids_list

In [34]:
def pad_prompt_response_ids(tokenizer, prompt_ids_list, response_ids_list):
    pad_id = tokenizer.pad_token_id

    sequences = []
    prompt_lens = []
    response_lens = []

    for prompt_ids, response_ids in zip(prompt_ids_list, response_ids_list):
        prompt_ids = prompt_ids.detach().cpu().long()
        response_ids = response_ids.detach().cpu().long()

        seq = torch.cat([prompt_ids, response_ids], dim=0)

        sequences.append(seq)
        prompt_lens.append(prompt_ids.numel())
        response_lens.append(response_ids.numel())

    max_len = max(seq.numel() for seq in sequences)
    B = len(sequences)

    input_ids = torch.full((B, max_len), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((B, max_len), dtype=torch.long)

    for i, seq in enumerate(sequences):
        L = seq.numel()
        input_ids[i, :L] = seq
        attention_mask[i, :L] = 1

    prompt_lens = torch.tensor(prompt_lens, dtype=torch.long)
    response_lens = torch.tensor(response_lens, dtype=torch.long)

    return input_ids, attention_mask, prompt_lens, response_lens

In [35]:
def response_logp_distributions(
    model,
    input_ids,
    attention_mask,
    prompt_lens,
    response_lens,
    require_grad=False,
    return_full_vocab=True,
):
    """
    Score only response tokens.

    Returns:
        response_logp_rows: [B, T, V] or None
        selected_logp:      [B, T]
        response_token_ids: [B, T]
        response_mask:      [B, T]
    """
    model_device = next(model.parameters()).device

    input_ids = input_ids.to(model_device)
    attention_mask = attention_mask.to(model_device)
    prompt_lens = prompt_lens.to(model_device).long()
    response_lens = response_lens.to(model_device).long()

    B, S = input_ids.shape
    T = int(response_lens.max().item())

    ctx = torch.enable_grad() if require_grad else torch.no_grad()

    with ctx:
        out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )

        logits = out.logits  # [B, S, V]

        # Standard causal-LM shift:
        # logits[:, k, :] predicts input_ids[:, k + 1]
        pred_logits = logits[:, :-1, :]      # [B, S-1, V]
        target_ids_all = input_ids[:, 1:]    # [B, S-1]

        logp_all = F.log_softmax(pred_logits, dim=-1)
        V = logp_all.shape[-1]

        # Response token y_t starts at original token position prompt_lens.
        # Since pred_logits row k predicts original token position k+1,
        # the prediction row for the first response token is prompt_lens - 1.
        t = torch.arange(T, device=model_device).unsqueeze(0)  # [1, T]

        response_pred_pos = (prompt_lens - 1).unsqueeze(-1) + t  # [B, T]

        response_mask = t < response_lens.unsqueeze(-1)  # [B, T]

        response_pred_pos = response_pred_pos.clamp(0, S - 2)

        response_token_ids = target_ids_all.gather(
            dim=1,
            index=response_pred_pos,
        )  # [B, T]

        response_logp_rows = logp_all.gather(
            dim=1,
            index=response_pred_pos.unsqueeze(-1).expand(B, T, V),
        )  # [B, T, V]

        selected_logp = response_logp_rows.gather(
            dim=2,
            index=response_token_ids.unsqueeze(-1),
        ).squeeze(-1)  # [B, T]

        selected_logp = selected_logp.masked_fill(~response_mask, 0.0)

        if not return_full_vocab:
            response_logp_rows = None

    return response_logp_rows, selected_logp, response_token_ids, response_mask

In [36]:
# ------------------------------------------------------------
# 4) Loss functions
# ------------------------------------------------------------

def opsd_full_vocab_forward_kl_loss(
    teacher_logp,
    student_logp,
    response_mask,
    pointwise_clip=None,
):
    """
    Eq. 6 / Eq. 8 with D = forward KL.

    Full-vocab loss:
        KL(p_T || p_S)
        = sum_v p_T(v) [log p_T(v) - log p_S(v)]
    """
    teacher_logp = teacher_logp.detach()
    teacher_prob = teacher_logp.exp().detach()

    pointwise = teacher_prob * (teacher_logp - student_logp)

    if pointwise_clip is not None:
        tau = torch.tensor(
            pointwise_clip,
            device=pointwise.device,
            dtype=pointwise.dtype,
        )
        pointwise = torch.minimum(pointwise, tau)

    per_token_kl = pointwise.sum(dim=-1)
    per_token_kl = per_token_kl.masked_fill(~response_mask, 0.0)

    mask = response_mask.to(per_token_kl.dtype)

    per_seq_loss = (
        (per_token_kl * mask).sum(dim=1)
        / mask.sum(dim=1).clamp_min(1)
    )

    loss = per_seq_loss.mean()

    stats = {
        "loss_name": "full_vocab_forward_kl",
        "mean_per_seq_loss": float(per_seq_loss.mean().detach().cpu()),
    }

    return loss, stats


In [37]:
def opsd_sampled_token_pg_loss(
    teacher_selected_logp,
    student_old_selected_logp,
    student_current_selected_logp,
    response_mask,
):
    """
    Eq. 9 sampled-token policy-gradient objective.

    A_n = log p_T(yhat_n) - log p_S_old(yhat_n)

    loss = - mean_n A_n * log p_S_current(yhat_n)

    A_n is detached.
    Gradients flow only through student_current_selected_logp.
    """
    advantage = (
        teacher_selected_logp
        - student_old_selected_logp
    ).detach()

    token_objective = advantage * student_current_selected_logp

    mask = response_mask.to(token_objective.dtype)

    per_seq_objective = (
        (token_objective * mask).sum(dim=1)
        / mask.sum(dim=1).clamp_min(1)
    )

    objective = per_seq_objective.mean()
    loss = -objective

    stats = {
        "loss_name": "sampled_token_pg",
        "mean_advantage": float(
            ((advantage * mask).sum() / mask.sum().clamp_min(1)).detach().cpu()
        ),
        "mean_per_seq_objective": float(per_seq_objective.mean().detach().cpu()),
    }

    return loss, stats

In [38]:
# ------------------------------------------------------------
# 5) One unified OPSD train step
# ------------------------------------------------------------

def opsd_train_step(
    student_model,
    teacher_model,
    tokenizer,
    batch_examples,
    optimizer,
    loss_type="full_kl",
    max_new_tokens=32,
    generation_temperature=1.0,
    generation_top_k=50,
    pointwise_clip=5.0,
):
    """
    Unified OPSD implementation.

    loss_type:
        "full_kl":
            Eq. 6 / Eq. 8, D = forward KL over full vocabulary.

        "sampled_pg":
            Eq. 9 sampled-token policy-gradient objective.
    """
    student_prompts = [
        make_student_prompt(ex["problem"])
        for ex in batch_examples
    ]
    print("\nThe Student Prompts are", student_prompts)

    teacher_prompts = [
        make_teacher_prompt(ex["problem"], ex["solution"])
        for ex in batch_examples
    ]
    print("\nThe Teacher Prompts are", teacher_prompts)

    # 1. Student on-policy rollout.
    response_ids_list = generate_student_rollouts(
        model=student_model,
        tokenizer=tokenizer,
        student_prompts=student_prompts,
        max_new_tokens=max_new_tokens,
        temperature=generation_temperature,
        top_k=generation_top_k,
    )

    # 2. Build student and teacher inputs using the exact same response IDs.
    student_prompt_ids_list = tokenize_prompts_no_padding(
        tokenizer,
        student_prompts,
    )

    teacher_prompt_ids_list = tokenize_prompts_no_padding(
        tokenizer,
        teacher_prompts,
    )

    s_input_ids, s_attention_mask, s_prompt_lens, s_response_lens = pad_prompt_response_ids(
        tokenizer,
        student_prompt_ids_list,
        response_ids_list,
    )

    t_input_ids, t_attention_mask, t_prompt_lens, t_response_lens = pad_prompt_response_ids(
        tokenizer,
        teacher_prompt_ids_list,
        response_ids_list,
    )

    s_input_ids = s_input_ids.to(device)
    s_attention_mask = s_attention_mask.to(device)
    s_prompt_lens = s_prompt_lens.to(device)
    s_response_lens = s_response_lens.to(device)

    t_input_ids = t_input_ids.to(device)
    t_attention_mask = t_attention_mask.to(device)
    t_prompt_lens = t_prompt_lens.to(device)
    t_response_lens = t_response_lens.to(device)

    if loss_type == "full_kl":
        # Teacher full-vocab distribution, frozen.
        with torch.no_grad():
            teacher_logp, teacher_selected_logp, _, teacher_mask = response_logp_distributions(
                model=teacher_model,
                input_ids=t_input_ids,
                attention_mask=t_attention_mask,
                prompt_lens=t_prompt_lens,
                response_lens=t_response_lens,
                require_grad=False,
                return_full_vocab=True,
            )

        # Student full-vocab distribution, trainable.
        student_model.train()

        student_logp, student_selected_logp, _, student_mask = response_logp_distributions(
            model=student_model,
            input_ids=s_input_ids,
            attention_mask=s_attention_mask,
            prompt_lens=s_prompt_lens,
            response_lens=s_response_lens,
            require_grad=True,
            return_full_vocab=True,
        )

        response_mask = teacher_mask & student_mask

        loss, loss_stats = opsd_full_vocab_forward_kl_loss(
            teacher_logp=teacher_logp,
            student_logp=student_logp,
            response_mask=response_mask,
            pointwise_clip=pointwise_clip,
        )

    elif loss_type == "sampled_pg":
        # Teacher selected-token logp, frozen.
        # Student-old selected-token logp, frozen constant for the advantage.
        with torch.no_grad():
            _, teacher_selected_logp, _, teacher_mask = response_logp_distributions(
                model=teacher_model,
                input_ids=t_input_ids,
                attention_mask=t_attention_mask,
                prompt_lens=t_prompt_lens,
                response_lens=t_response_lens,
                require_grad=False,
                return_full_vocab=False,
            )

            _, student_old_selected_logp, _, student_old_mask = response_logp_distributions(
                model=student_model,
                input_ids=s_input_ids,
                attention_mask=s_attention_mask,
                prompt_lens=s_prompt_lens,
                response_lens=s_response_lens,
                require_grad=False,
                return_full_vocab=False,
            )

        # Student-current selected-token logp, trainable.
        student_model.train()

        _, student_current_selected_logp, _, student_current_mask = response_logp_distributions(
            model=student_model,
            input_ids=s_input_ids,
            attention_mask=s_attention_mask,
            prompt_lens=s_prompt_lens,
            response_lens=s_response_lens,
            require_grad=True,
            return_full_vocab=False,
        )

        response_mask = teacher_mask & student_old_mask & student_current_mask

        loss, loss_stats = opsd_sampled_token_pg_loss(
            teacher_selected_logp=teacher_selected_logp,
            student_old_selected_logp=student_old_selected_logp,
            student_current_selected_logp=student_current_selected_logp,
            response_mask=response_mask,
        )

        student_selected_logp = student_current_selected_logp

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        mask = response_mask.to(torch.float32)

        completions = [
            tokenizer.decode(
                ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )
            for ids in response_ids_list
        ]

        if loss_type == "full_kl":
            mean_student_sample_logp = (
                (student_selected_logp * mask).sum()
                / mask.sum().clamp_min(1)
            )
            mean_teacher_sample_logp = (
                (teacher_selected_logp * mask).sum()
                / mask.sum().clamp_min(1)
            )
        else:
            mean_student_sample_logp = (
                (student_selected_logp * mask).sum()
                / mask.sum().clamp_min(1)
            )
            mean_teacher_sample_logp = (
                (teacher_selected_logp * mask).sum()
                / mask.sum().clamp_min(1)
            )

        stats = {
            "loss": float(loss.detach().cpu()),
            "loss_type": loss_type,
            "mean_student_sample_logp": float(mean_student_sample_logp.detach().cpu()),
            "mean_teacher_sample_logp": float(mean_teacher_sample_logp.detach().cpu()),
            "response_lens": s_response_lens.detach().cpu().tolist(),
            "completions": completions,
        }

        stats.update(loss_stats)

    return stats

In [39]:
# ------------------------------------------------------------
# 1) Data
# ------------------------------------------------------------

examples = [
    {
        "problem": "Find the derivative of f(x) = 3x^2 + 2x - 5 at x = 2.",
        "solution": "First compute f'(x) = 6x + 2. Then f'(2) = 6*2 + 2 = 14.",
    },
    {
        "problem": "Solve for a: 2a + 5 = 17.",
        "solution": "Subtract 5 from both sides to get 2a = 12. Divide by 2, so a = 6.",
    },
    {
        "problem": "A rectangle has length 9 and width 4. What is its area?",
        "solution": "The area of a rectangle is length times width. So area = 9 * 4 = 36.",
    },
    {
        "problem": "A train travels 60 miles in 2 hours. What is its average speed?",
        "solution": "Average speed equals distance divided by time. So speed = 60 / 2 = 30 miles per hour.",
    },
]

In [40]:
# ------------------------------------------------------------
# 6) Run one OPSD objective on two batches
# ------------------------------------------------------------

loss_type = "full_kl"
# loss_type = "sampled_pg"

opsd_batches = [
    examples[:2],
    examples[2:],
]

print("\n" + "#" * 80)
print(f"RUNNING OPSD OBJECTIVE: {loss_type}")
print("#" * 80)

for step, batch_examples in enumerate(opsd_batches, start=1):

    print("\n" + "=" * 80)
    print(f"OPSD BATCH {step} / batch_size = {len(batch_examples)}")
    print("=" * 80)

    for i, ex in enumerate(batch_examples):
        print(f"\nExample {i}")
        print("Problem:", ex["problem"])
        print("Reference solution:", ex["solution"])

    stats = opsd_train_step(
        student_model=student_model,
        teacher_model=teacher_model,
        tokenizer=tokenizer,
        batch_examples=batch_examples,
        optimizer=optimizer,
        loss_type=loss_type,
        max_new_tokens=32,
        generation_temperature=1.0,
        generation_top_k=50,
        pointwise_clip=5.0,
    )

    print("\nStudent-generated rollouts:")
    for i, completion in enumerate(stats["completions"]):
        print(f"  rollout {i}: {repr(completion)}")

    print("\nStats:")
    for k, v in stats.items():
        if k != "completions":
            print(f"{k}: {v}")


################################################################################
RUNNING OPSD OBJECTIVE: full_kl
################################################################################

OPSD BATCH 1 / batch_size = 2

Example 0
Problem: Find the derivative of f(x) = 3x^2 + 2x - 5 at x = 2.
Reference solution: First compute f'(x) = 6x + 2. Then f'(2) = 6*2 + 2 = 14.

Example 1
Problem: Solve for a: 2a + 5 = 17.
Reference solution: Subtract 5 from both sides to get 2a = 12. Divide by 2, so a = 6.

The Student Prompts are ['Problem: Find the derivative of f(x) = 3x^2 + 2x - 5 at x = 2.\nAnswer:', 'Problem: Solve for a: 2a + 5 = 17.\nAnswer:']

The Teacher Prompts are ["Problem: Find the derivative of f(x) = 3x^2 + 2x - 5 at x = 2.\nHere is a reference solution:\nFirst compute f'(x) = 6x + 2. Then f'(2) = 6*2 + 2 = 14.\nAfter understanding the reference solution, please try to solve this problem using your own approach below:\nAnswer:", 'Problem: Solve for a: 2a + 5 = 17.\nHere is